# Week2 예습과제 - 순환 신경망과 트랜스포머

- 교재: (파이토치 트랜스포머를 활용한) 자연어 처리와 컴퓨터 비전 심층학습
- 범위: 6장 순환 신경망 (p.304~335) / 7장 트랜스포머 (p.357~387)
- 논문을 읽기 전에 필요한 개념을 먼저 정리하는 것이 목적임

## 실행 환경

- 런타임 유형을 GPU로 바꾼 뒤 위에서부터 순서대로 실행함
- 형태소 분석과 모델 학습에 시간이 걸리므로 6장 실습 전체는 30분 정도 잡아야 함
- 교재의 `pip install torchdata torchtext portalocker`는 넣지 않음. torchtext가 배포 중단돼 현재 파이토치 버전과 맞지 않고, Multi30k 원본 주소도 닫혀 있음. 7장 실습 코드는 다음 주차 복습과제 범위이므로 코드 블록으로만 적어 둠

In [ ]:
!pip install -q Korpora konlpy gensim

## fastText의 OOV 처리

Word2Vec은 단어 하나를 통째로 학습하므로 단어 사전에 없는 단어는 임베딩을 만들 수 없음. fastText는 단어를 하위 단어로 나누어 학습하므로 사전에 없는 단어도 하위 단어의 임베딩을 합해 표현할 수 있음.

예를 들어 '사랑해요'가 사전에 없더라도 '사랑', '랑해', '해요' 같은 하위 단어로 분해되므로, 다른 단어에 등장했던 '사랑'이나 '해요'를 통해 임베딩을 계산할 수 있음. 한국어처럼 형태적 구조를 갖는 언어에서 특히 효과적임.

교재 예제 6.17은 6.5절에서 학습해 둔 fastText 모델을 그대로 쓰는 코드임. 이 노트북에는 그 모델이 없으므로 같은 말뭉치로 작은 모델을 먼저 학습함.

In [ ]:
from Korpora import Korpora
from konlpy.tag import Okt
from gensim.models import FastText

corpus = Korpora.load("nsmc")
tokenizer = Okt()

# 하위 단어 임베딩을 확인하는 용도이므로 앞쪽 일부만 사용한다.
sample_tokens = [tokenizer.morphs(review) for review in corpus.test.texts[:5000]]

fastText = FastText(
    sentences=sample_tokens,
    vector_size=128,
    window=5,
    min_count=3,
    sg=1,
    epochs=5,
    min_n=2,
    max_n=5,
)
print("어휘 사전 크기:", len(fastText.wv.index_to_key))

### 예제 6.17 fastText OOV 처리

In [ ]:
oov_token = "사랑해요"
oov_vector = fastText.wv[oov_token]

print(oov_token in fastText.wv.index_to_key)
print(fastText.wv.most_similar(oov_vector, topn=5))

`wv.index_to_key`는 학습된 단어 사전임. '사랑해요'는 사전에 없으므로 첫 줄은 False가 나옴. 그런데도 임베딩을 구할 수 있고 유사한 단어까지 찾을 수 있는 것이 fastText의 특징임.

# 1. 순환 신경망

## 1) 순환 신경망 (RNN)

순서가 있는 연속적인 데이터를 처리하는 신경망임. 자연어, 시계열, 음성처럼 현재 시점의 데이터가 앞선 시점의 데이터와 독립적이지 않은 경우에 사용함.

자연어가 대표적인 예임. '금요일이 지나면 _'이라는 문장에서 빈칸에 '주말'이나 '토요일'이 올 것을 앞 단어들의 패턴으로 예측할 수 있음. 즉 $t$번째 단어는 $t-1$번째까지의 단어에 영향을 받아 결정됨.

순환 신경망은 각 시점마다 은닉 상태를 저장하고, 이를 다음 시점으로 넘김. 입력을 받아 은닉 상태와 출력값을 계산하는 노드를 셀이라고 함.

$$h_t = \sigma_h(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$

$x_t$는 현재 입력, $h_{t-1}$은 이전 시점의 은닉 상태임. $W_{hh}$는 이전 은닉 상태에 대한 가중치, $W_{xh}$는 입력값에 대한 가중치, $b_h$는 편향임.

$$y_t = \sigma_y(W_{hy} h_t + b_y)$$

현재 은닉 상태를 변환해 출력값을 구함. 이전 정보를 다음 시점으로 전달하면서 시퀀스의 패턴을 학습하는 구조임.

### 일대다 / 다대일 / 다대다

**일대다**는 하나의 입력 시퀀스에 대해 여러 개의 출력값을 생성하는 구조임. 이미지 한 장을 받아 설명 문장을 만드는 이미지 캡셔닝이 여기에 해당함. 출력 시퀀스의 길이를 미리 알 수 없으므로 길이를 예측하는 모델이 함께 필요함.

**다대일**은 여러 입력에 대해 하나의 출력값을 생성하는 구조임. 문장을 읽고 긍정·부정을 판단하는 감성 분류, 문장 분류, 자연어 추론에 사용함.

**다대다**는 입력과 출력이 모두 시퀀스인 구조임. 기계 번역, 음성 인식 등에 쓰이며 입력과 출력의 길이가 다를 수 있음. 길이가 다르면 패딩을 추가하거나 잘라 내는 전처리를 함.

다대다 구조는 시퀀스-시퀀스 구조로 이뤄져 있음. 입력 시퀀스를 처리하는 인코더가 고정 크기의 벡터를 만들고, 디코더가 그 벡터를 받아 출력 시퀀스를 생성함.

### 양방향 순환 신경망 / 다중 순환 신경망

**양방향 순환 신경망**은 입력을 순방향과 역방향으로 모두 처리함. 현재 위치를 해석할 때 앞의 문맥뿐 아니라 뒤의 문맥도 이용할 수 있음.

"인생은 B와 _ 사이의 C다"라는 문장에서 빈칸 앞만 봐서는 어떤 단어가 올지 알기 어렵지만, 뒤의 '사이의 C이다'를 보면 'D'가 들어갈 것을 알 수 있음. 대부분의 연속형 데이터는 이후 시점의 데이터와도 상관관계가 크므로 양쪽 정보를 모두 고려함.

**다중 순환 신경망**은 여러 순환 신경망 층을 쌓은 구조임. 각 층의 출력값이 다음 층으로 전달되어 처리됨. 층이 많아질수록 다양한 특징을 추출할 수 있지만 학습 시간이 오래 걸리고 기울기 소실이 발생할 가능성도 높아짐.

### 순환 신경망 클래스

파이토치가 제공하는 클래스임. 아래는 매개변수를 보여 주는 형태이므로 그대로 실행하지 않음.

```python
rnn = torch.nn.RNN(
    input_size,
    hidden_size,
    num_layers=1,
    nonlinearity="tanh",
    bias=False,
    batch_first=True,
    dropout=0,
    bidirectional=False
)
```

- `input_size`는 입력 특성 크기, `hidden_size`는 은닉 상태 크기임
- `num_layers`는 층 수이며 2 이상이면 다중 순환 신경망이 됨
- `nonlinearity`는 활성화 함수로 `tanh`와 `relu`를 쓸 수 있음
- `bias`는 편향 사용 여부임
- `batch_first`가 참이면 입력을 [배치 크기, 시퀀스 길이, 입력 특성 크기]로 전달하고, 거짓이면 [시퀀스 길이, 배치 크기, 입력 특성 크기]로 전달함
- `dropout`은 과대적합 방지를 위한 드롭아웃 확률, `bidirectional`은 양방향 처리 여부임

### 예제 6.18 양방향 다층 신경망

In [ ]:
import torch
from torch import nn


input_size = 128
ouput_size = 256
num_layers = 3
bidirectional = True

model = nn.RNN(
    input_size=input_size,
    hidden_size=ouput_size,
    num_layers=num_layers,
    nonlinearity="tanh",
    batch_first=True,
    bidirectional=bidirectional,
)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.rand(num_layers * (int(bidirectional) + 1), batch_size, ouput_size)

outputs, hidden = model(inputs, h_0)
print(outputs.shape)
print(hidden.shape)

`outputs`는 모든 시점의 출력값이고 `hidden`은 최종 은닉 상태임.

출력값은 [배치 크기, 시퀀스 길이, (양방향 여부 + 1) × 은닉 상태 크기]이므로 [4, 6, 512]가 됨. 최종 은닉 상태는 초기 은닉 상태와 같은 차원인 [계층 수 × 양방향 여부 + 1, 배치 크기, 은닉 상태 크기]이므로 [6, 4, 256]이 됨.

배치 우선 매개변수에 따라 차원의 형태가 달라지므로 설정에 주의해야 함.

## 2) 장단기 메모리 (LSTM)

순환 신경망은 시간적으로 연속된 데이터를 다룰 수 있지만, 앞선 시점의 정보를 끊임없이 반영하기 때문에 학습 데이터가 길어지면 앞서 학습한 정보가 충분히 전달되지 않음. 이를 **장기 의존성 문제**라고 함. 활성화 함수로 쓰는 하이퍼볼릭 탄젠트나 ReLU의 특성 때문에 역전파 과정에서 기울기 소실이나 폭주가 발생할 가능성도 있음.

장단기 메모리는 1997년 셉 호흐라이터와 위르겐 슈미트후버가 제안한 알고리즘으로, **메모리 셀**과 **게이트**를 도입해 이 문제를 해결함.

- 셀 상태는 정보를 저장하고 유지하는 역할이며 출력 게이트와 망각 게이트에 의해 제어됨
- 망각 게이트는 이전 셀 상태에서 어떤 정보를 삭제할지 결정함
- 입력(기억) 게이트는 새로운 정보를 어떤 부분에 추가할지 결정함
- 출력 게이트는 셀 상태 중 어떤 부분을 출력할지 결정함

세 게이트는 모두 시그모이드를 활성화 함수로 사용하므로 출력이 0과 1 사이임. 1에 가까우면 정보를 많이 통과시키고 0에 가까우면 적게 통과시킴.

### 게이트 연산

**망각 게이트**는 이전 시점의 은닉 상태와 현재 입력으로 계산함.

$$f_t = \sigma\left(W_x^{(f)} x_t + W_h^{(f)} h_{t-1} + b^{(f)}\right)$$

출력값이 1에 가까울수록 이전 정보를 유지하고, 0에 가까울수록 삭제함. 정확히 1이면 아무 정보도 삭제하지 않고, 0이면 모든 정보를 삭제함.

**기억 게이트**는 새로 추가할 후보 정보 $g_t$와 그 반영량 $i_t$로 구성됨.

$$g_t = \tanh\left(W_x^{(g)} x_t + W_h^{(g)} h_{t-1} + b^{(g)}\right)$$

$$i_t = \sigma\left(W_x^{(i)} x_t + W_h^{(i)} h_{t-1} + b^{(i)}\right)$$

$g_t$는 하이퍼볼릭 탄젠트를 쓰므로 $[-1, 1]$ 범위이고, $i_t$는 시그모이드를 쓰므로 $[0, 1]$ 범위임. $i_t$가 1에 가까울수록 새 정보를 많이 기억함.

$$c_t = f_t \odot c_{t-1} + g_t \odot i_t$$

이전 기억에서 유지할 부분과 새로 추가할 부분을 더해 셀 상태를 갱신함. $\odot$는 같은 위치의 원소끼리 곱하는 아다마르 곱임.

**출력 게이트**는 갱신된 셀 상태에서 무엇을 출력할지 결정함.

$$o_t = \sigma\left(W_x^{(o)} x_t + W_h^{(o)} h_{t-1} + b^{(o)}\right)$$

$$h_t = o_t \odot \tanh(c_t)$$

셀 상태에 하이퍼볼릭 탄젠트를 적용한 값과 출력 게이트를 아다마르 곱하여 현재 은닉 상태를 구함. 이 방식으로 현재 은닉 상태가 이전 은닉 상태의 정보를 얼마나 반영할지 조절함.

### 장단기 메모리 클래스

```python
lstm = torch.nn.LSTM(
    input_size,
    hidden_size,
    num_layers=1,
    bias=False,
    batch_first=True,
    dropout=0,
    bidirectional=False,
    proj_size=0
)
```

순환 신경망 클래스와 거의 같지만 활성화 함수를 명확하게 정의해 사용하므로 `nonlinearity` 매개변수가 없음.

`proj_size`는 출력에 적용하는 **선형 투사**의 크기임. 0보다 크면 은닉 상태를 선형 투사로 다른 차원에 매핑하므로 출력 차원을 줄이거나 다른 차원으로 변환할 수 있음. 0이면 은닉 상태의 차원을 그대로 유지함. 투사 크기는 은닉 상태 크기보다 작은 값으로 설정해야 함.

### 예제 6.19 양방향 다층 장단기 메모리

In [ ]:
import torch
from torch import nn

input_size = 128
ouput_size = 256
num_layers = 3
bidirectional = True
proj_size = 64

model = nn.LSTM(
    input_size=input_size,
    hidden_size=ouput_size,
    num_layers=num_layers,
    batch_first=True,
    bidirectional=bidirectional,
    proj_size=proj_size,
)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.rand(
    num_layers * (int(bidirectional) + 1),
    batch_size,
    proj_size if proj_size > 0 else ouput_size,
)
c_0 = torch.rand(num_layers * (int(bidirectional) + 1), batch_size, ouput_size)

outputs, (h_n, c_n) = model(inputs, (h_0, c_0))

print(outputs.shape)
print(h_n.shape)
print(c_n.shape)

입력값과 함께 초기 은닉 상태 `h_0`, 초기 메모리 셀 상태 `c_0`를 전달함. 결과로 출력값과 최종 은닉 상태 `h_n`, 최종 메모리 셀 상태 `c_n`을 반환함.

출력값은 선형 투사가 적용되므로 [4, 6, 2×64]인 [4, 6, 128]이 됨. 최종 은닉 상태도 투사 크기를 따라 [6, 4, 64]가 되지만, 메모리 셀은 투사를 적용하지 않으므로 [6, 4, 256]을 유지함.

은닉 상태와 메모리 셀 상태를 튜플로 묶어 사용하므로 순환 신경망과 형태가 다름.

## 3) 모델 실습 - 문장 긍/부정 분류

순환 신경망과 장단기 메모리로 네이버 영화 리뷰의 긍정·부정을 분류하는 모델을 만듦. 구조는 임베딩 계층 → 순환 신경망 → 드롭아웃 → 분류기임.

임베딩 계층은 [어휘 사전 크기, 임베딩 벡터 크기]의 순람표임. 입력 텍스트를 정수 인코딩한 뒤 해당 색인의 임베딩을 가져오는 역할임. 초깃값으로 무작위 값을 할당하고 학습을 통해 최적화하는 방법과, 사전 학습된 임베딩 벡터를 가져와 사용하는 방법이 있음. 두 가지를 모두 다룸.

### 예제 6.20 문장 분류 모델

In [ ]:
from torch import nn


class SentenceClassifier(nn.Module):
    def __init__(
        self,
        n_vocab,
        hidden_dim,
        embedding_dim,
        n_layers,
        dropout=0.5,
        bidirectional=True,
        model_type="lstm"
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=n_vocab,
            embedding_dim=embedding_dim,
            padding_idx=0
        )
        if model_type == "rnn":
            self.model = nn.RNN(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True,
            )
        elif model_type == "lstm":
            self.model = nn.LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True,
            )

        if bidirectional:
            self.classifier = nn.Linear(hidden_dim * 2, 1)
        else:
            self.classifier = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        output, _ = self.model(embeddings)
        last_output = output[:, -1, :]
        last_output = self.dropout(last_output)
        logits = self.classifier(last_output)
        return logits

`model_type`으로 순환 신경망과 장단기 메모리를 선택함. 양방향이면 출력 특성 크기가 두 배가 되므로 분류기의 입력 크기도 `hidden_dim` × 2로 맞춤.

순방향 메서드에서는 입력을 임베딩 계층과 모델에 통과시키고, 마지막 시점의 결괏값만 `[:, -1, :]`로 가져와 드롭아웃과 분류기에 전달함.

### 예제 6.21 데이터세트 불러오기

네이버 영화 리뷰 감정 분석 데이터세트를 사용함. 데이터 크기가 작은 `corpus.test`를 가져와 학습용 90%, 테스트용 10%로 나눔.

In [ ]:
import pandas as pd
from Korpora import Korpora


corpus = Korpora.load("nsmc")
corpus_df = pd.DataFrame(corpus.test)

train = corpus_df.sample(frac=0.9, random_state=42)
test = corpus_df.drop(train.index)

print(train.head(5).to_markdown())
print("Training Data Size :", len(train))
print("Testing Data Size :", len(test))

### 예제 6.22 데이터 토큰화 및 단어 사전 구축

Okt 토크나이저로 문장을 형태소 단위로 나누고 자주 등장하는 토큰으로 단어 사전을 구성함. 문장 길이를 맞추기 위한 `<pad>`와 사전에 없는 토큰을 나타내는 `<unk>`를 특수 토큰으로 추가함.

형태소 분석에 몇 분 걸림.

In [ ]:
from konlpy.tag import Okt
from collections import Counter


def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab


tokenizer = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]

vocab = build_vocab(corpus=train_tokens, n_vocab=5000, special_tokens=["<pad>", "<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

단어 5,000개에 특수 토큰 2개를 더해 어휘 사전 크기는 5,002가 됨.

### 예제 6.23 정수 인코딩 및 패딩

토큰을 단어 사전의 정수로 변환함. 최대 길이보다 긴 문장은 자르고 짧은 문장은 뒤에 `<pad>`를 채워 길이를 맞춤. 최대 길이가 너무 크면 입력 행렬이 커져 자원을 많이 쓰고, 너무 짧으면 문장을 제대로 반영하지 못함.

In [ ]:
import numpy as np


def pad_sequences(sequences, max_length, pad_value):
    result = list()
    for sequence in sequences:
        sequence = sequence[:max_length]
        pad_length = max_length - len(sequence)
        padded_sequence = sequence + [pad_value] * pad_length
        result.append(padded_sequence)
    return np.asarray(result)


unk_id = token_to_id["<unk>"]
train_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in train_tokens
]
test_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in test_tokens
]

max_length = 32
pad_id = token_to_id["<pad>"]
train_ids = pad_sequences(train_ids, max_length, pad_id)
test_ids = pad_sequences(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])

### 예제 6.24 데이터로더 적용

정수 인코딩한 문장과 레이블을 텐서로 바꿔 데이터세트로 묶음. 학습 데이터는 섞어서 불러오고 테스트 데이터는 순서대로 불러옴.

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader


train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch.tensor(train.label.values, dtype=torch.float32)
test_labels = torch.tensor(test.label.values, dtype=torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

### 예제 6.25 손실 함수와 최적화 함수 정의

긍정·부정 두 가지를 분류하므로 이진 교차 엔트로피를 사용함. `BCEWithLogitsLoss`는 `BCELoss`와 시그모이드를 결합한 형태라서 모델 출력에 시그모이드를 따로 적용하지 않아도 됨.

최적화 함수는 RMSProp을 적용함. 모든 기울기를 누적하지 않고 지수 가중 이동 평균을 사용해 학습률을 조절함. 기울기가 큰 구간에서는 학습률을 줄여 발산을 막고, 작은 구간에서는 학습률을 키워 지역 최솟값에 빠지는 것을 방지함.

In [ ]:
from torch import optim


n_vocab = len(token_to_id)
hidden_dim = 64
embedding_dim = 128
n_layers = 2

device = "cuda" if torch.cuda.is_available() else "cpu"
classifier = SentenceClassifier(
    n_vocab=n_vocab, hidden_dim=hidden_dim, embedding_dim=embedding_dim, n_layers=n_layers
).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr=0.001)

### 예제 6.26 모델 학습 및 테스트

학습에서는 손실을 계산한 뒤 역전파와 가중치 갱신을 수행함. 테스트에서는 시그모이드 결과가 0.5보다 큰지로 예측 레이블을 정하고 정확도를 계산함.

In [ ]:
def train(model, datasets, criterion, optimizer, device, interval):
    model.train()
    losses = list()

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % interval == 0:
            print(f"Train Loss {step} : {np.mean(losses)}")


def test(model, datasets, criterion, device):
    model.eval()
    losses = list()
    corrects = list()

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())
        yhat = torch.sigmoid(logits) > .5
        corrects.extend(
            torch.eq(yhat, labels).cpu().tolist()
        )

    print(f"Val Loss : {np.mean(losses)}, Val Accuracy : {np.mean(corrects)}")


epochs = 5
interval = 500

for epoch in range(epochs):
    train(classifier, train_loader, criterion, optimizer, device, interval)
    test(classifier, test_loader, criterion, device)

에폭마다 검증 손실과 검증 정확도를 확인함. 학습이 진행되면서 손실이 줄고 정확도가 오르는지를 보면 됨.

### 예제 6.27 학습된 모델로부터 임베딩 추출

문장 분류 모델을 학습하면 임베딩 계층의 가중치도 함께 최적화됨. 학습된 가중치에서 각 단어의 벡터를 가져와 토큰별 임베딩으로 사용할 수 있음.

다만 긍/부정 분류는 임베딩 계층보다 순환 신경망의 연산이 더 중요하게 동작하므로, 모델이 복잡할수록 임베딩이 토큰의 의미 정보를 학습하기는 어려움.

In [ ]:
token_to_embedding = dict()
embedding_matrix = classifier.embedding.weight.detach().cpu().numpy()

for word, emb in zip(vocab, embedding_matrix):
    token_to_embedding[word] = emb

token = vocab[1000]
print(token, token_to_embedding[token])

### 예제 6.28 사전 학습된 모델로 임베딩 계층 초기화

이번에는 사전 학습된 임베딩 값을 초깃값으로 적용함. 교재는 6.4절에서 같은 말뭉치로 학습해 둔 Word2Vec 모델을 `../models/word2vec.model`에서 불러옴. 이 노트북에는 그 파일이 없으므로 앞에서 만든 `train_tokens`로 같은 자리에 모델을 만들어 저장함.

넘파이 배열로 초기화하되 `<pad>`와 `<unk>` 토큰은 초기화에서 제외함.

In [ ]:
import os
from gensim.models import Word2Vec

os.makedirs("../models", exist_ok=True)

# 교재 6.4절에서 학습한 모델을 대신해 동일한 말뭉치로 학습한다.
if not os.path.exists("../models/word2vec.model"):
    w2v = Word2Vec(
        sentences=train_tokens,
        vector_size=embedding_dim,
        window=5,
        min_count=1,
        sg=1,
        epochs=3,
        max_final_vocab=10000,
    )
    w2v.save("../models/word2vec.model")

In [ ]:
from gensim.models import Word2Vec


word2vec = Word2Vec.load("../models/word2vec.model")
init_embeddings = np.zeros((n_vocab, embedding_dim))

for index, token in id_to_token.items():
    if token not in ["<pad>", "<unk>"]:
        init_embeddings[index] = word2vec.wv[token]

embedding_layer = nn.Embedding.from_pretrained(
    torch.tensor(init_embeddings, dtype=torch.float32)
)

### 예제 6.29 사전 학습된 임베딩 계층 적용

교재는 예제 6.20에서 바뀌는 부분만 `...`로 표시함. 그대로 두면 실행되지 않으므로 여기서는 `pretrained_embedding` 매개변수를 반영한 전체 클래스로 다시 작성함.

`pretrained_embedding`이 `None`이 아니면 전달된 값으로 임베딩 계층을 초기화하고, `None`이면 기존 방식으로 임베딩을 생성함.

In [ ]:
class SentenceClassifier(nn.Module):
    def __init__(
        self,
        n_vocab,
        hidden_dim,
        embedding_dim,
        n_layers,
        dropout=0.5,
        bidirectional=True,
        model_type="lstm",
        pretrained_embedding=None
    ):
        super().__init__()

        if pretrained_embedding is not None:
            self.embedding = nn.Embedding.from_pretrained(
                torch.tensor(pretrained_embedding, dtype=torch.float32)
            )
        else:
            self.embedding = nn.Embedding(
                num_embeddings=n_vocab,
                embedding_dim=embedding_dim,
                padding_idx=0
            )

        if model_type == "rnn":
            self.model = nn.RNN(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True,
            )
        elif model_type == "lstm":
            self.model = nn.LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True,
            )

        if bidirectional:
            self.classifier = nn.Linear(hidden_dim * 2, 1)
        else:
            self.classifier = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        output, _ = self.model(embeddings)
        last_output = output[:, -1, :]
        last_output = self.dropout(last_output)
        logits = self.classifier(last_output)
        return logits

### 예제 6.30 사전 학습된 임베딩을 사용한 모델 학습

In [ ]:
classifier = SentenceClassifier(
    n_vocab=n_vocab, hidden_dim=hidden_dim, embedding_dim=embedding_dim,
    n_layers=n_layers, pretrained_embedding=init_embeddings
).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr=0.001)

epochs = 5
interval = 500

for epoch in range(epochs):
    train(classifier, train_loader, criterion, optimizer, device, interval)
    test(classifier, test_loader, criterion, device)

사전 학습된 임베딩을 쓰는 것은 성능을 개선할 수 있는 방법 중 하나임. 다만 학습 데이터가 충분하다면 모델의 목적에 맞게 임베딩 계층을 새로 학습하는 것이 더 나은 결과를 줄 수도 있음. 한국어처럼 형태 정보가 중요한 언어에서는 그 점을 고려한 임베딩 방법이 유리할 수 있으므로, 데이터 특성에 맞춰 선택해야 함.

# 2. 트랜스포머

## 1) Transformer

2017년 「Attention is All You Need」 논문에서 소개된 아키텍처임. 기존 순환 신경망처럼 순차적으로 처리하지 않고 입력 시퀀스를 병렬로 처리함. 긴 시퀀스에서 순환 신경망보다 훨씬 빠르고 효율적임.

순차 처리나 반복 연결에 의존하지 않고 입력 토큰 간의 관계를 직접 처리하는 **셀프 어텐션**을 기반으로 하기 때문임. 재귀나 합성곱 연산 없이 입력 토큰 간의 관계를 직접 모델링할 수 있음.

인코더는 소스 시퀀스를 임베딩해 고차원 벡터로 변환하고, 디코더는 인코더의 출력을 입력으로 받아 출력 시퀀스를 생성함. 어텐션 메커니즘이 두 시퀀스 사이의 상관관계를 계산해 중요한 정보에 집중함.

| 모델 | 학습 구조 | 학습 방법 | 학습 방향성 |
| --- | --- | --- | --- |
| BERT | 인코더 | 오토 인코딩 | 양방향 |
| GPT | 디코더 | 자기 회귀 | 단방향 |
| BART | 인코더+디코더 | 오토 인코딩+자기 회귀 | 양방향+단방향 |
| ELECTRA | 인코더+판별기 | 오토 인코딩+대체 토큰 탐지 | 양방향 |
| T5 | 인코더+디코더 | 오토 인코딩+자기 회귀 | 양방향 |

오토 인코딩은 문장의 일부를 빈칸 토큰으로 만들고 어떤 단어가 적절할지 예측하는 방식임. 빈칸의 양쪽 토큰을 모두 참조하므로 양방향 구조를 가지며 이를 인코더라고 함. 자기 회귀는 이전 단어들이 주어졌을 때 다음 단어를 맞히는 방식이며, 예측되는 단어의 왼쪽 토큰만 참조하므로 단방향 구조를 가지고 이를 디코더라고 함.

## 2) 입력 임베딩과 위치 인코딩

입력 시퀀스의 각 단어는 임베딩되어 벡터 형태로 변환됨. 트랜스포머는 시퀀스를 병렬 구조로 처리하기 때문에 단어의 순서 정보를 따로 제공해야 함. 그래서 위치 정보를 임베딩 벡터에 더하는 **위치 인코딩**을 사용함.

$$PE_{(pos,\,2i)} = \sin\left(pos / 10000^{2i/d_{model}}\right)$$

$$PE_{(pos,\,2i+1)} = \cos\left(pos / 10000^{2i/d_{model}}\right)$$

$pos$는 입력 시퀀스에서 단어의 위치, $i$는 임베딩 벡터의 차원 인덱스임. 차원 인덱스가 짝수면 sin, 홀수면 cos를 적용함. $1/10000^{2i/d_{model}}$은 각도 정보로 변환하기 위한 스케일링 인자로, 각 위치마다 주기적인 신호를 생성함.

토큰의 위치마다 서로 다른 값을 더해 주므로 모델이 순서 정보를 학습할 수 있게 됨.

### 예제 7.1 위치 인코딩

In [ ]:
import math
import torch
from torch import nn
from matplotlib import pyplot as plt


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[: x.size(0)]
        return self.dropout(x)


encoding = PositionalEncoding(d_model=128, max_len=50)

plt.pcolormesh(encoding.pe.numpy().squeeze(), cmap="RdBu")
plt.xlabel("Embedding Dimension")
plt.xlim((0, 128))
plt.ylabel("Position")
plt.colorbar()
plt.show()

`d_model`은 입력 임베딩 차원, `max_len`은 최대 시퀀스 길이임. `pe`의 텐서 차원은 [50, 1, 128]이 되며 [최대 시퀀스, 1, 임베딩 차원]을 의미함. 출력 그림을 보면 위치별 임베딩 차원이 주기적인 값으로 구성되는 것을 확인할 수 있음.

`register_buffer`로 등록하면 모델이 매개변수로 갱신하지 않음.

## 3) 특수 토큰

트랜스포머는 단어 토큰 이외의 특수 토큰을 활용해 문장을 표현함. 입력 시퀀스의 시작과 끝을 나타내거나 마스킹 영역으로 사용함.

- **BOS**는 문장의 시작을 나타냄
- **EOS**는 문장의 끝을 나타냄
- **UNK**는 어휘 사전에 없는 단어, 즉 모르는 단어를 의미함
- **PAD**는 모든 문장을 일정한 길이로 맞추기 위해 빈 공간을 채우는 토큰임

이렇게 생성된 문장 토큰 배열을 어휘 사전에 등장하는 위치에 원-핫 인코딩으로 표현함. 어휘 사전의 크기가 $V$, 임베딩 차원이 $d$이면 원-핫 벡터 $[1, V]$가 임베딩 행렬 $[V, d]$에 의해 $[1, d]$로 변환됨. 문장 $N$개가 최대 $S$개의 토큰 길이를 가질 때 $[N, S, V]$의 원-핫 벡터 텐서는 $[N, S, d]$의 임베딩 텐서로 변환됨.

## 4) 트랜스포머 인코더

위치 인코딩이 적용된 소스 데이터의 입력 임베딩을 입력받음. 멀티 헤드 어텐션 단계에서 입력 텐서 차원이 $[N, S, d]$라면 선형 변환을 통해 세 개의 임베딩 벡터를 생성함. 각각 **쿼리(Q)**, **키(K)**, **값(V)** 벡터임.

- 쿼리는 현재 시점에서 참조하고자 하는 정보의 위치를 나타내는 벡터임. 인코더의 각 시점마다 생성됨
- 키는 쿼리와 비교되는 대상으로, 쿼리를 제외한 입력 시퀀스에서 탐색되는 벡터임
- 값은 쿼리와 키로 생성된 어텐션 스코어를 얼마나 반영할지 설정하는 가중치 역할임

$$score(v^q, v^k) = \text{softmax}\left(\frac{(v^q)^\top \cdot v^k}{\sqrt{d}}\right)$$

쿼리와 키 벡터를 내적해 어텐션 스코어를 구하고, 이 스코어를 $\sqrt{d}$로 나눠 보정함. 벡터 차원이 커질 때 스코어값이 같이 커지는 문제를 완화하기 위함임. 보정된 스코어를 소프트맥스로 확률로 만들고 값 벡터와 내적해 셀프 어텐션 벡터를 생성함.

**셀프 어텐션**은 같은 입력 시퀀스 안에서 관계를 계산하는 방식임. 각 단어가 다른 단어를 참고하면서 문맥이 반영된 표현으로 바뀜.

**멀티 헤드**는 이러한 셀프 어텐션을 여러 번 수행해 여러 개의 헤드를 만드는 것임. 입력값이 $[N, S, d]$일 때 $k$개의 헤드를 쓰면 $[N, k, S, d/k]$ 형태가 되고, 각 헤드가 독립적으로 어텐션을 수행한 뒤 결과를 임베딩 차원 축으로 다시 병합해 $[N, S, d]$로 출력함.

**덧셈 & 정규화**는 멀티 헤드 어텐션을 통과하기 이전의 입력값과 이후의 출력값을 더해 학습 시 발생하는 기울기 소실을 완화함. 그리고 임베딩 차원 축으로 계층 정규화를 적용함.

**순방향 신경망**은 선형 임베딩과 ReLU로 이뤄진 인공 신경망이나 1차원 합성곱으로 구성됨. 이 과정에서 산출된 임베딩 벡터를 더 고도화함. 이후 다시 덧셈 & 정규화를 수행함.

인코더는 이러한 블록 여러 개로 구성되며, 마지막 블록의 출력 벡터가 디코더의 멀티 헤드 어텐션에서 키와 값으로 사용됨.

## 5) 트랜스포머 디코더

위치 인코딩이 적용된 타깃 데이터의 입력 임베딩을 입력받음. 구조는 인코더와 비슷하지만 멀티 헤드 어텐션 모듈이 **인과성**을 반영한 **마스크 멀티 헤드 어텐션**으로 대체됨.

어텐션 스코어 맵을 계산할 때 첫 번째 쿼리 벡터가 첫 번째 키 벡터만 바라보게 마스크를 씌우고, 두 번째 쿼리 벡터는 첫 번째와 두 번째 키 벡터만 바라보게 마스크를 씌움. 이렇게 하면 현재 위치 이전의 단어들만 참조할 수 있게 되어 인과성이 보장됨.

마스크 영역에는 $-\infty$를 더해 줌. 소프트맥스를 계산하면 해당 영역의 어텐션 가중치가 0에 가까워지므로 미래 토큰을 참조하지 않게 됨.

디코더의 두 번째 어텐션에서는 타깃 데이터가 쿼리 벡터로 사용되고, 인코더의 소스 데이터가 키와 값 벡터로 사용됨. 출력 문장을 만들면서 소스 문장의 정보를 참조하는 과정임.

마지막 디코더 블록의 출력 텐서 $[N, S, d]$에 선형 변환과 소프트맥스를 적용해 타깃 시퀀스 위치마다 예측 확률을 계산함.

디코더는 타깃 데이터를 추론할 때 토큰을 순차적으로 생성함. 아직 생성되지 않은 빈 공간은 PAD 토큰으로 채움. 예를 들어 'ChatGPT는 트랜스포머 모델로 이뤄져 있다'를 추론하면 `[BOS]`만 있는 입력에서 시작해 한 번에 한 토큰씩 채워 나가고, 마지막에 `[EOS]`가 나오면 종료함.

## 6) 모델 실습 (다음 주차 복습과제 범위)

교재는 파이토치의 트랜스포머 모델로 영어-독일어 번역 모델을 구성함. 학습 데이터는 약 30,000개의 영어-독일어 **병렬 말뭉치**인 Multi30k 데이터세트를 사용함. 병렬 말뭉치는 2개 국어 이상의 번역된 문서를 모은 말뭉치임.

교재는 아래와 같이 라이브러리를 설치함.

```
pip install torchdata torchtext portalocker
```

- 토치 데이터는 대규모 데이터세트를 불러오고 변환·배치하는 API를 제공함
- 토치 텍스트는 사전 처리와 데이터세트 관리를 위한 도구를 제공함
- 포르타락커는 파일 락을 관리하는 라이브러리로, Multi30k를 내려받고 압축을 해제하는 과정에서 내부적으로 사용됨

> 이 노트북에서는 예제 7.2와 7.5~7.8을 실행하지 않음. torchtext가 배포 중단되어 현재 코랩의 파이토치 버전과 함께 설치되지 않고, 설치하더라도 Multi30k 원본 주소가 닫혀 있어 데이터를 내려받을 수 없음. 해당 코드는 다음 주차 복습과제 범위이므로 개념만 정리하고 코드 블록으로 남겨 둠.

### 예제 7.2 데이터세트 다운로드 및 전처리

```python
from torchtext.datasets import Multi30k
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator


def generate_tokens(text_iter, language):
    language_index = {SRC_LANGUAGE: 0, TGT_LANGUAGE: 1}

    for text in text_iter:
        yield token_transform[language](text[language_index[language]])


SRC_LANGUAGE = "de"
TGT_LANGUAGE = "en"
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ["<unk>", "<pad>", "<bos>", "<eos>"]

token_transform = {
    SRC_LANGUAGE: get_tokenizer("spacy", language="de_core_news_sm"),
    TGT_LANGUAGE: get_tokenizer("spacy", language="en_core_web_sm"),
}

vocab_transform = {}
for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    train_iter = Multi30k(split="train", language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
    vocab_transform[language] = build_vocab_from_iterator(
        generate_tokens(train_iter, language),
        min_freq=1,
        specials=special_symbols,
        special_first=True,
    )

for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    vocab_transform[language].set_default_index(UNK_IDX)
```

`token_transform`은 언어별 토크나이저를, `vocab_transform`은 언어별 어휘 사전을 저장함. `get_tokenizer`는 spaCy에 사전 학습된 토크나이저를 가져오는 함수로, `python -m spacy download de_core_news_sm` 형태로 모델을 따로 내려받아야 함.

`min_freq`는 단어 사전에 포함할 최소 빈도수임. `special_first=True`로 특수 토큰을 사전 맨 앞에 추가하고, `set_default_index`로 사전에 없는 토큰에 `<unk>` 인덱스를 할당함.

### 예제 7.3 트랜스포머 모델 구성

`TokenEmbedding`으로 소스와 타깃을 각각 임베딩하고 위치 인코딩을 더함. 트랜스포머 블록을 통과한 출력은 `generator`를 통해 타깃 어휘 사전에 대한 로짓으로 바뀜.

이 셀은 클래스 정의만 하므로 데이터 없이 실행됨.

In [ ]:
import math
import torch
from torch import nn


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[: x.size(0)]
        return self.dropout(x)


class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, emb_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.emb_size = emb_size

    def forward(self, tokens):
        return self.embedding(tokens.long()) * math.sqrt(self.emb_size)


class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        num_encoder_layers,
        num_decoder_layers,
        emb_size,
        max_len,
        nhead,
        src_vocab_size,
        tgt_vocab_size,
        dim_feedforward,
        dropout=0.1,
    ):
        super().__init__()
        self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(
            d_model=emb_size, max_len=max_len, dropout=dropout
        )
        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
        )
        self.generator = nn.Linear(emb_size, tgt_vocab_size)

    def forward(
        self,
        src,
        trg,
        src_mask,
        tgt_mask,
        src_padding_mask,
        tgt_padding_mask,
        memory_key_padding_mask,
    ):
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(trg))
        outs = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            memory_mask=None,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )
        return self.generator(outs)

    def encode(self, src, src_mask):
        return self.transformer.encoder(
            self.positional_encoding(self.src_tok_emb(src)), src_mask
        )

    def decode(self, tgt, memory, tgt_mask):
        return self.transformer.decoder(
            self.positional_encoding(self.tgt_tok_emb(tgt)), memory, tgt_mask
        )

### 트랜스포머 클래스

```python
transformer = torch.nn.Transformer(
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1,
    activation=torch.nn.functional.relu,
    layer_norm_eps=1e-05,
)
```

- `d_model`은 임베딩 차원이며 입력과 출력 차원의 크기임
- `nhead`는 멀티 헤드 어텐션의 헤드 개수임. 많을수록 병렬 처리 능력이 커지지만 모델 매개변수 수도 늘어남
- `num_encoder_layers`, `num_decoder_layers`는 인코더와 디코더의 계층 수임. 많을수록 복잡한 문제를 해결할 수 있으나 과대적합될 수 있음
- `dim_feedforward`는 순방향 신경망의 은닉층 크기임
- `dropout`은 드롭아웃 비율, `activation`은 순방향 신경망의 활성화 함수, `layer_norm_eps`는 계층 정규화에서 분모에 더해지는 입실론임

### 트랜스포머 순방향 메서드

```python
output = transformer.forward(
    src,
    tgt,
    src_mask=None,
    tgt_mask=None,
    memory_mask=None,
    src_key_padding_mask=None,
    tgt_key_padding_mask=None,
    memory_key_padding_mask=None,
)
```

- `src`, `tgt`는 [소스(타깃) 시퀀스 길이, 배치 크기, 임베딩 차원] 형태임
- `src_mask`, `tgt_mask`는 [소스(타깃) 시퀀스 길이, 시퀀스 길이] 형태임. 값이 0이면 모든 입력 단어가 동일한 가중치를 갖고, 1이면 가중치가 0으로 설정돼 어텐션 연산이 수행되지 않음. $-\infty$면 어텐션 연산 결과에 0으로 가중치가 부여돼 마스킹된 위치의 정보를 무시하게 됨
- `memory_mask`는 인코더 출력의 마스크로 [타깃 시퀀스 길이, 소스 시퀀스 길이] 형태임
- `key_padding_mask`는 패딩 토큰이 위치한 부분을 가리는 이진 마스크임. 패딩은 실제 의미를 갖지 않으므로 해당 위치의 어텐션 가중치를 0으로 만듦

### 예제 7.4 트랜스포머 모델 구조

교재는 예제 7.2에서 만든 `vocab_transform`으로 어휘 사전 크기를 넘김. 여기서는 예제 7.2를 실행하지 않으므로 구조만 확인할 수 있도록 어휘 사전 크기를 임의의 값으로 대신함.

In [ ]:
from torch import optim


BATCH_SIZE = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 교재에서는 len(vocab_transform[SRC_LANGUAGE]) 값을 사용한다.
src_vocab_size = 19214
tgt_vocab_size = 10837

model = Seq2SeqTransformer(
    num_encoder_layers=3,
    num_decoder_layers=3,
    emb_size=512,
    max_len=512,
    nhead=8,
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    dim_feedforward=512,
).to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=1).to(DEVICE)
optimizer = optim.Adam(model.parameters())

for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("└", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("│  └", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("│  │  └", sssub_name)

출력에서 확인할 수 있듯이 입력 임베딩(`src_tok_emb`, `tgt_tok_emb`), 위치 인코딩(`positional_encoding`), 트랜스포머 블록(`transformer`), 로짓 생성(`generator`)으로 구성됨. 인코더와 디코더가 각각 세 개(0, 1, 2)의 계층으로 구성되는 것도 볼 수 있음.

손실 함수는 교차 엔트로피를 적용하되 `ignore_index`에 PAD 인덱스를 지정함. 패딩 토큰은 학습에 사용되지 않으므로 해당 클래스에 대한 손실은 계산되지 않음.

### 예제 7.5 배치 데이터 생성

```python
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence


def sequential_transforms(*transforms):
    def func(txt_input):
        for transform in transforms:
            txt_input = transform(txt_input)
        return txt_input
    return func


def input_transform(token_ids):
    return torch.cat(
        (torch.tensor([BOS_IDX]), torch.tensor(token_ids), torch.tensor([EOS_IDX]))
    )


def collator(batch):
    src_batch, tgt_batch = [], []
    for src_sample, tgt_sample in batch:
        src_batch.append(text_transform[SRC_LANGUAGE](src_sample.rstrip("\n")))
        tgt_batch.append(text_transform[TGT_LANGUAGE](tgt_sample.rstrip("\n")))

    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX)
    return src_batch, tgt_batch


text_transform = {}
for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    text_transform[language] = sequential_transforms(
        token_transform[language], vocab_transform[language], input_transform
    )

data_iter = Multi30k(split="valid", language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
dataloader = DataLoader(data_iter, batch_size=BATCH_SIZE, collate_fn=collator)
source_tensor, target_tensor = next(iter(dataloader))
```

`sequential_transforms`는 여러 전처리 함수를 순서대로 적용하는 함수를 반환함. 토큰화 → 정수 인코딩 → BOS·EOS 추가 순으로 적용됨.

`collator`는 배치 단위로 데이터를 처리함. `rstrip("\n")`으로 개행 문자를 제거하고, `pad_sequence`로 길이를 맞춤. 소스와 타깃이 각각 패딩되므로 최대 시퀀스 길이가 서로 다를 수 있음. 출력 차원은 [소스(타깃) 시퀀스 길이, 배치 크기]임.

### 예제 7.6 어텐션 마스크 생성

```python
def generate_square_subsequent_mask(s):
    mask = (torch.triu(torch.ones((s, s), device=DEVICE)) == 1).transpose(0, 1)
    mask = (
        mask.float()
        .masked_fill(mask == 0, float("-inf"))
        .masked_fill(mask == 1, float(0.0))
    )
    return mask


def create_mask(src, tgt):
    src_seq_len = src.shape[0]
    tgt_seq_len = tgt.shape[0]

    tgt_mask = generate_square_subsequent_mask(tgt_seq_len)
    src_mask = torch.zeros((src_seq_len, src_seq_len), device=DEVICE).type(torch.bool)

    src_padding_mask = (src == PAD_IDX).transpose(0, 1)
    tgt_padding_mask = (tgt == PAD_IDX).transpose(0, 1)
    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask


target_input = target_tensor[:-1, :]
target_out = target_tensor[1:, :]

source_mask, target_mask, source_padding_mask, target_padding_mask = create_mask(
    source_tensor, target_input
)
```

`generate_square_subsequent_mask`는 `torch.ones`로 1로 채워진 행렬을 만든 뒤 `torch.triu`로 상삼각행렬을 만들고 전치시킴. 0인 값은 $-\infty$로, 1인 값은 0.0으로 채워 어텐션 연산에 적용함. 0.0으로 설정된 값은 셀프 어텐션에 참조되는 시퀀스를 가리키며, $-\infty$ 값은 어텐션 스코어가 0에 수렴하므로 해당 타깃 입력 시퀀스를 제외시킴.

패딩 마스크를 만들기 전에 타깃 데이터의 입력값(`target_input`)과 출력값(`target_out`)을 토큰 순서로 한 칸 시프트함. 이전 토큰들이 주어졌을 때 다음 토큰을 예측하게 하기 위함임.

인과 마스크는 미래 정보 차단, 패딩 마스크는 빈 위치 제외라는 차이가 있음.

### 예제 7.7 모델 학습 및 평가

```python
def run(model, optimizer, criterion, split):
    model.train() if split == "train" else model.eval()
    data_iter = Multi30k(split=split, language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
    dataloader = DataLoader(data_iter, batch_size=BATCH_SIZE, collate_fn=collator)

    losses = 0
    for source_batch, target_batch in dataloader:
        source_batch = source_batch.to(DEVICE)
        target_batch = target_batch.to(DEVICE)

        target_input = target_batch[:-1, :]
        target_output = target_batch[1:, :]

        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(
            source_batch, target_input
        )

        logits = model(
            src=source_batch,
            trg=target_input,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            src_padding_mask=src_padding_mask,
            tgt_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask,
        )

        optimizer.zero_grad()
        loss = criterion(logits.reshape(-1, logits.shape[-1]), target_output.reshape(-1))
        if split == "train":
            loss.backward()
            optimizer.step()
        losses += loss.item()

    return losses / len(list(dataloader))
```

소스 문장과 타깃의 앞부분을 입력받아 다음 타깃 토큰의 로짓을 계산함. 정답과의 교차 엔트로피를 구하고, 학습 단계에서만 역전파와 가중치 갱신을 수행함.

### 예제 7.8 트랜스포머 모델 번역 결과

```python
def greedy_decode(model, source_tensor, source_mask, max_len, start_symbol):
    source_tensor = source_tensor.to(DEVICE)
    source_mask = source_mask.to(DEVICE)

    memory = model.encode(source_tensor, source_mask)
    ys = torch.ones(1, 1).fill_(start_symbol).type(torch.long).to(DEVICE)
    for i in range(max_len - 1):
        memory = memory.to(DEVICE)
        target_mask = generate_square_subsequent_mask(ys.size(0))
        target_mask = target_mask.type(torch.bool).to(DEVICE)

        out = model.decode(ys, memory, target_mask)
        out = out.transpose(0, 1)
        prob = model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.item()

        ys = torch.cat(
            [ys, torch.ones(1, 1).type_as(source_tensor.data).fill_(next_word)], dim=0
        )
        if next_word == EOS_IDX:
            break
    return ys
```

**그리디 디코딩**은 현재 시점에서 가장 높은 점수의 토큰 하나를 선택하는 방식임. 선택한 토큰을 다시 입력에 붙여 다음 토큰을 생성함.

소스 문장을 인코딩해 `memory`를 만든 뒤 디코더가 이를 참고함. BOS부터 시작해 생성한 토큰을 이어 붙이고, EOS가 나오거나 최대 길이에 도달하면 종료함. 교재 예제에서는 소스 토큰 수에 5를 더해 최대 생성 길이를 정함.

# 정리

- 순환 신경망은 은닉 상태를 다음 시점으로 넘기며 순서 정보를 처리함. 다만 시퀀스가 길어지면 앞선 정보가 충분히 전달되지 않는 장기 의존성 문제가 생김
- 장단기 메모리는 셀 상태와 세 개의 게이트로 무엇을 기억하고 지울지 조절해 그 문제를 완화함
- 양방향 구조는 뒤 문맥까지 보고, 다중 구조는 층을 쌓아 더 복잡한 특징을 학습함. 대신 학습 시간과 기울기 소실 위험이 커짐
- 문장 분류 실습에서는 임베딩 계층을 무작위로 초기화하는 방법과 Word2Vec으로 초기화하는 방법을 비교함. 사전 학습 임베딩이 늘 유리한 것은 아니고 데이터 양과 목적에 따라 달라짐
- 트랜스포머는 순환 구조 없이 셀프 어텐션만으로 토큰 간 관계를 직접 계산함. 병렬 처리가 가능해 긴 시퀀스에서 유리함
- 병렬로 처리하기 때문에 순서 정보가 사라지므로 위치 인코딩을 임베딩에 더해 줌
- 인코더는 양방향으로 문맥을 보고, 디코더는 마스크로 미래 토큰을 가려 인과성을 지킴. BERT는 인코더, GPT는 디코더를 중심으로 사용함